In [19]:
import geopandas as gpd
from geopandas.geodataframe import GeoDataFrame

candidates: GeoDataFrame = gpd.read_parquet("data/17_NYC_Solar_Candidates_June21_Final.parquet")

In [20]:
print(candidates["MEAN"].describe())
print(len(candidates))
print(len(candidates["doitt_id"].unique()))

count    247.000000
mean      15.786512
std        9.743427
min      -27.213971
25%        8.708328
50%       12.503585
75%       23.589871
max       43.106960
Name: MEAN, dtype: float64
12946
12946


In [27]:
candidates["LandUse"].value_counts()

LandUse
05    7516
06    5430
Name: count, dtype: int64

## Electricity generated and utility-bill savings (10-building subsample)

`MEAN` values (-27 to 43) don't match Wh/m² for June 21, so `MEAN` is used only as a **relative** solar score, anchored to a standard NYC irradiance. (Note: `MEAN` correlates 0.95 with `ground_ele`, so it may be ground elevation rather than solar potential.)

- `irradiance = min(1400 kWh/m²/yr * MEAN / median(MEAN), 1600)` (median over all rows with valid MEAN; capped at a feasible NYC maximum)
- `kWh/yr = AREA (sq ft -> m²) * usable roof fraction * irradiance * panel efficiency * performance ratio`
- `building use = AREA * NumFloors * kWh/sqft/yr for its LandUse` (05 office ~16, 06 industrial ~12)
- `$ saved/yr = min(solar, building use) * retail rate + excess solar * export credit`

In [21]:
# Assumptions (edit as needed)
NYC_IRRADIANCE_KWH_M2_YR = 1400  # typical annual insolation for NYC rooftops
MAX_IRRADIANCE_KWH_M2_YR = 1600  # feasible upper bound in NYC (optimally tilted, unshaded panel)
PANEL_EFFICIENCY = 0.20
PERFORMANCE_RATIO = 0.80  # inverter, wiring, soiling losses
USABLE_ROOF_FRACTION = 0.3  # setbacks, HVAC, access paths
RATE_USD_PER_KWH = 0.28  # approx. Con Edison blended rate (solar that offsets building use)
EXPORT_CREDIT_USD_PER_KWH = 0.10  # approx. credit for excess solar sent to the grid
SQFT_TO_M2 = 0.092903

# Annual building electricity use per sq ft of floor area, by MapPLUTO LandUse code
KWH_PER_SQFT_BY_LANDUSE = {
    "05": 16.0,  # Commercial & Office
    "06": 12.0,  # Industrial & Manufacturing
}

N_SAMPLE = 10
SEED = 42

In [22]:
# Keep buildings with a usable solar score and roof area (drops nulls and negative MEAN)
valid = candidates[(candidates["MEAN"] > 0) & (candidates["AREA"] > 0)]
mean_median = valid["MEAN"].median()

sample = valid.sample(N_SAMPLE, random_state=SEED).copy()
print(f"{len(valid)} valid buildings, median MEAN = {mean_median:.2f}, sampled {len(sample)}")

246 valid buildings, median MEAN = 12.52, sampled 10


In [23]:
# Solar generation
sample["area_m2"] = sample["AREA"] * SQFT_TO_M2
sample["irradiance_kwh_m2_yr"] = (
    NYC_IRRADIANCE_KWH_M2_YR * sample["MEAN"] / mean_median
).clip(upper=MAX_IRRADIANCE_KWH_M2_YR)
sample["kwh_per_year"] = (
    sample["area_m2"]
    * USABLE_ROOF_FRACTION
    * sample["irradiance_kwh_m2_yr"]
    * PANEL_EFFICIENCY
    * PERFORMANCE_RATIO
)

# Building electricity use: footprint x floors x usage intensity for its land use
sample["floor_area_sqft"] = sample["AREA"] * sample["NumFloors"]
sample["building_kwh_per_year"] = (
    sample["floor_area_sqft"] * sample["LandUse"].map(KWH_PER_SQFT_BY_LANDUSE)
)

# Solar up to building use offsets the bill at retail; any excess earns the export credit
sample["kwh_offset"] = sample[["kwh_per_year", "building_kwh_per_year"]].min(axis=1)
sample["kwh_exported"] = sample["kwh_per_year"] - sample["kwh_offset"]
sample["usd_saved_per_year"] = (
    sample["kwh_offset"] * RATE_USD_PER_KWH
    + sample["kwh_exported"] * EXPORT_CREDIT_USD_PER_KWH
)

In [26]:
results = sample[
    [
        "doitt_id", "LandUse", "NumFloors", "Roof_Elev_ft", "AREA", "MEAN",
        "irradiance_kwh_m2_yr", "kwh_per_year", "building_kwh_per_year",
        "kwh_offset", "kwh_exported", "usd_saved_per_year",
    ]
].sort_values("usd_saved_per_year", ascending=False)

print(
    f"Total: {results['kwh_per_year'].sum():,.0f} kWh/yr generated, "
    f"{results['kwh_offset'].sum() / results['building_kwh_per_year'].sum():.1%} of building use offset, "
    f"${results['usd_saved_per_year'].sum():,.0f}/yr saved"
)
results = results.rename(columns={"AREA": "area_sqft", "MEAN": "TEMP_MEAN_ADJUSTED_SOLAR_SCORE"})
results.round(2)

Total: 1,883,096 kWh/yr generated, 2.1% of building use offset, $527,267/yr saved


,doitt_id,LandUse,NumFloors,Roof_Elev_ft,area_sqft,TEMP_MEAN_ADJUSTED_SOLAR_SCORE,irradiance_kwh_m2_yr,kwh_per_year,building_kwh_per_year,kwh_offset,kwh_exported,usd_saved_per_year
9114,709036.0,05,40.0,589.48,74776.0,13.94,1559.56,520036.84,47856640.0,520036.84,0.0,145610.31
462,1288410.0,05,4.0,80.00,65248.0,12.72,1422.42,413872.67,4175872.0,413872.67,0.0,115884.35
7819,324915.0,05,32.0,504.19,43050.0,30.55,1600.00,307159.61,22041600.0,307159.61,0.0,86004.69
5292,562580.0,06,1.0,29.59,35930.0,13.54,1514.23,242616.66,431160.0,242616.66,0.0,67932.66
9305,149015.0,06,1.0,27.00,22339.0,12.16,1360.05,135484.29,268068.0,135484.29,0.0,37935.60
12125,406519.0,05,34.0,448.25,21175.0,7.90,883.97,83470.14,11519200.0,83470.14,0.0,23371.64
4058,4912.0,05,20.0,304.01,8210.0,21.73,1600.00,58577.94,2627200.0,58577.94,0.0,16401.82
10030,38704.0,06,1.0,24.52,11850.0,9.64,1078.12,56971.46,142200.0,56971.46,0.0,15952.01
1781,135661.0,05,5.0,138.82,6415.0,31.05,1600.00,45770.71,513200.0,45770.71,0.0,12815.80
528,939904.0,06,1.0,28.90,6434.0,5.96,666.96,19135.89,77208.0,19135.89,0.0,5358.05
